# Capítulo 9 — Otimização de inferência

**Livro:** começa na página **730** do PDF.

**Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Métricas de desempenho de inferência · PDF p. 742

#### Exercício: TTFT, TPOT e tokens por segundo

Com `stream=True`, meça para uma resposta de ~150 tokens: **TTFT** (tempo até o 1º token), **TPOT** (tempo médio por token de saída) e **tokens por segundo**. Use `contar_tokens()` na resposta completa.

In [ ]:
import time
inicio = time.perf_counter(); primeiro = None; texto = ""
for parte in cliente.chat.completions.create(model=MODELO, stream=True, max_tokens=150,
        messages=[{"role": "user", "content": "Explique em detalhes o que é cache de prompt."}]):
    if parte.choices and parte.choices[0].delta.content:
        primeiro = primeiro or time.perf_counter()
        texto += parte.choices[0].delta.content
fim = time.perf_counter()
n = contar_tokens(texto)
print(f"TTFT {primeiro - inicio:.2f}s · TPOT {(fim - primeiro) / max(n - 1, 1) * 1000:.0f} ms/token · "
      f"{n / (fim - inicio):.1f} tokens/s ({n} tokens)")

### Otimização do serviço de inferência · PDF p. 792

#### Exercício: Requisições em paralelo

Faça 5 perguntas curtas **uma depois da outra** e depois **em paralelo** (`concurrent.futures.ThreadPoolExecutor`). Compare o tempo total.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
perguntas = [f"Dê um sinônimo para a palavra '{p}'. Responda só a palavra." for p in
             ["rápido", "estoque", "pedido", "atraso", "qualidade"]]

t = time.perf_counter(); seq = [perguntar(p, max_tokens=5) for p in perguntas]; t_seq = time.perf_counter() - t
t = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as ex:
    par = list(ex.map(lambda p: perguntar(p, max_tokens=5), perguntas))
t_par = time.perf_counter() - t
print(f"Sequencial: {t_seq:.1f}s · Paralelo: {t_par:.1f}s")

## Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou